In [1]:
import torch

# Autograd

**GOAL:** How PyTorch constructs a computation graph and applies the chain rule during backward propagation.

## Experiment

1. Create a tensor.
2. Run the following 4 operations on the tensor, in the same order:
  - multiply a random tensor (same tensor, with requires_grad=True)
  - multiply by a scalar (same scalar, with requires_grad=True)
  - sum the elements of the results to get the output.

3. The expected output is pre decided (not necessary to be the same as the sum from the above step)
4. Check the grad_fn for each tensor
5. perform back propagation using tensor.backward() and then manually calculating gradients.

In [2]:
# Input tensors
x = torch.rand((10000))

In [8]:
# Output
y = 10

In [9]:
x

tensor([0.6537, 0.5391, 0.1207,  ..., 0.9862, 0.4286, 0.6671])

In [10]:
# random tensor
random_tensor = torch.rand((10000), requires_grad=True)
# random scalar
random_scalar = torch.rand((1, 1), requires_grad=True)

In [11]:
random_tensor

tensor([0.6653, 0.3995, 0.3480,  ..., 0.1164, 0.4413, 0.1887],
       requires_grad=True)

In [12]:
random_scalar

tensor([[0.2401]], requires_grad=True)

### Operations

In [13]:
x_1 = x * random_tensor

In [14]:
x_1

tensor([0.4349, 0.2154, 0.0420,  ..., 0.1148, 0.1891, 0.1259],
       grad_fn=<MulBackward0>)

In [15]:
x_2 = x_1 * random_scalar

In [16]:
x_2

tensor([[0.1044, 0.0517, 0.0101,  ..., 0.0275, 0.0454, 0.0302]],
       grad_fn=<MulBackward0>)

### Sum

In [17]:
x_3 = x_2.sum()

In [18]:
x_3

tensor(604.7047, grad_fn=<SumBackward0>)

In [33]:
loss = x_3 - y

In [34]:
loss

tensor(594.7047, grad_fn=<SubBackward0>)

### Checking grad_fn

In [26]:
x.grad_fn == None

True

In [29]:
x_1.grad_fn, x_1.grad_fn.next_functions

(<MulBackward0 at 0x7e83c53d2c80>,
 ((None, 0), (<AccumulateGrad at 0x7e83c539a440>, 0)))

In [30]:
x_2.grad_fn, x_2.grad_fn.next_functions

(<MulBackward0 at 0x7e83c53d3400>,
 ((<MulBackward0 at 0x7e83c53d2c80>, 0),
  (<AccumulateGrad at 0x7e83c539a8c0>, 0)))

In [31]:
x_3.grad_fn, x_3.grad_fn.next_functions

(<SumBackward0 at 0x7e83c53d36d0>, ((<MulBackward0 at 0x7e83c53d3400>, 0),))

In [35]:
loss.grad_fn, loss.grad_fn.next_functions

(<SubBackward0 at 0x7e83c53995a0>,
 ((<SumBackward0 at 0x7e83c53d36d0>, 0), (None, 0)))

### Manual Backpropagation


**NOTE:** For ease of typing, I will use the following abbreviations for the partial derivatives:

- dx_1 = dL/d(random_tensor)
- dx_2 = dL/d(random_scalar)

loss = x2.sum() - y

loss = (x_1 * random_scalar).sum() - y

dL/d(random_scalar) = (x_1 * (d(random_scalar)/d(random_scalar))).sum() - y * (d(1)/d(random_scalar))

dL/d(random_scalar) = x_1.sum() - 0

dL/d(random_scalar) = (x * random_tensor).sum() = dx_2

In [113]:
dx_2 = (x*random_tensor).sum()

loss = x2.sum() - y

loss = (x_1 * random_scalar).sum() - y

loss = (x * random_tensor * random_scalar).sum() - y

dL/d(random_tensor) = (x * random_scalar * (d(random_tensor)/d(random_tensor))) - y * (d(1)/d(random_tensor))

--- .sum() is removed because random_tensor is a vector

dL/d(random_tensor) = (x * random_scalar) - 0

dL/d(random_tensor) = (x * random_scalar) = dx_1

In [116]:
dx_1 = x*random_scalar

In [117]:
dx_1, dx_2

(tensor([[0.8578, 1.7156, 2.5734, 3.4312]], grad_fn=<MulBackward0>),
 tensor(5.5258, grad_fn=<SumBackward0>))

### Autograd

In [119]:
loss.backward()

In [122]:
loss

tensor(-5.2600, grad_fn=<SubBackward0>)

In [123]:
random_tensor.grad, random_scalar.grad

(tensor([0.8578, 1.7156, 2.5734, 3.4312]), tensor([[5.5258]]))

## requires_grad experiment

1. create 2 tensors: one with requires_grad = True, other without
2. Add those tensors and store in a new variable
3. loss = torch.prod(new_variable)
4. perform loss.backward()
5. check the grad for the 2 original tensors

Prediction: the one with requires_grad=true will have a grad and the other one won't

In [36]:
# input tensors
x1, x2 = torch.rand((10000), requires_grad=True), torch.rand((10000))

In [37]:
y = x1 + x2

In [38]:
y

tensor([0.8814, 1.7107, 1.7066,  ..., 0.4433, 0.8262, 1.2175],
       grad_fn=<AddBackward0>)

In [39]:
loss = torch.prod(y)

In [40]:
loss

tensor(0., grad_fn=<ProdBackward0>)

In [41]:
loss.backward()

In [42]:
x1.grad, x2.grad

(tensor([0., 0., 0.,  ..., 0., 0., 0.]), None)

Observation: Predictions were correct. The tensor without requires_grad=true is not tracked by PyTorch

## leaf tensors experiment

1. create 1 tensor with requires_grad = True
2. Add the tensor with a scalar and store in a new variable
3. loss = torch.prod(new_variable)
4. check if each of the tensors are leaf tensors with .is_leaf.
5. perform loss.backward()
6. check the grad for all the tensors
7. repeat 1 to 4
8. run .retain_grad() on the new_variable tensor
9. run 5 and 6


In [43]:
x = torch.rand((10000), requires_grad=True)

In [44]:
x1 = x * 2

In [45]:
x, x1

(tensor([0.9459, 0.5840, 0.9700,  ..., 0.4807, 0.8254, 0.4639],
        requires_grad=True),
 tensor([1.8917, 1.1681, 1.9400,  ..., 0.9615, 1.6508, 0.9278],
        grad_fn=<MulBackward0>))

In [46]:
loss = torch.prod(x1)

In [47]:
loss

tensor(0., grad_fn=<ProdBackward0>)

In [48]:
x.is_leaf, x1.is_leaf

(True, False)

In [49]:
loss.backward()

In [50]:
x.grad, x1.grad

/tmp/ipykernel_1203/1245656431.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  x.grad, x1.grad


(tensor([0., 0., 0.,  ..., 0., 0., 0.]), None)

In [51]:
x = torch.rand((10000), requires_grad=True)

In [52]:
x1 = x*2

In [53]:
loss = torch.prod(x1)

In [54]:
x.is_leaf, x1.is_leaf

(True, False)

In [55]:
x1.retain_grad()

In [56]:
loss.backward()

In [57]:
x.grad, x1.grad

(tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]))

Observation: to save gradients for non-leaf tensors, we need to run .retain_grad() for that tensor

## Gradient accumulation experiment

1. create a tensor with requires_grad=true
2. multiply it with 5
3. loss = product.sum()
4. run loss.backward()
4. run 2 to 4 in a loop for 5 times
5. check the grad of the tensor

In [62]:
x = torch.rand((10000), requires_grad=True)

In [63]:
x

tensor([0.5471, 0.5378, 0.3682,  ..., 0.5095, 0.8240, 0.9541],
       requires_grad=True)

In [64]:
for i in range(5):
  y = x*5
  loss = y.sum()
  loss.backward()
  print(f"Iter {i+1}", x.grad)

Iter 1 tensor([5., 5., 5.,  ..., 5., 5., 5.])
Iter 2 tensor([10., 10., 10.,  ..., 10., 10., 10.])
Iter 3 tensor([15., 15., 15.,  ..., 15., 15., 15.])
Iter 4 tensor([20., 20., 20.,  ..., 20., 20., 20.])
Iter 5 tensor([25., 25., 25.,  ..., 25., 25., 25.])


Observations: The gradients are accumulated unless specifically discarded